# Notebook 03 — Spatial Join

**Purpose:** Convert fire detections to spatial points and join them to district polygons so every detection gains a `state` and `district` label.

**Input:** `data/processed/fire_filtered.csv`, `data/processed/Punjab_Haryana_districts.gpkg`  
**Output:** `data/processed/fire_with_districts.csv`

In [13]:
import pandas as pd
import geopandas as gpd

# Load filtered fire data 
fire = pd.read_csv('../Data/Processed/fire_filtered.csv')
print(f"Fire rows loaded: {len(fire):,}")
print(f"Columns: {fire.columns.tolist()}")


Fire rows loaded: 1,242,007
Columns: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type', 'month', 'year']


In [14]:
# Convert to GeoDataFrame (tabular → spatial)
# Each row becomes a POINT geometry built from latitude/longitude.
# CRS EPSG:4326 = WGS-84, the standard GPS coordinate system.
fire_gdf = gpd.GeoDataFrame(
    fire,
    geometry=gpd.points_from_xy(fire['longitude'], fire['latitude']),
    crs='EPSG:4326'
)
print(f"Converted to GeoDataFrame: {len(fire_gdf):,} points")


✅ Converted to GeoDataFrame: 1,242,007 points


In [15]:
# Load district polygons 
districts = gpd.read_file('../Data/Processed/Punjab_Haryana_districts.gpkg')
districts = districts.to_crs('EPSG:4326')   # ensure matching CRS
print(f"District polygons loaded: {len(districts)}")
print(districts[['NAME_1', 'NAME_2']].head())


District polygons loaded: 43
    NAME_1     NAME_2
0  Haryana     Ambala
1  Haryana    Bhiwani
2  Haryana  Faridabad
3  Haryana  Fatehabad
4  Haryana   Gurugram


In [16]:
#  Spatial join — point-in-polygon 
# how='inner'        → keep only fire points that fall inside a polygon
# predicate='within' → standard point-in-polygon test
joined = gpd.sjoin(
    fire_gdf,
    districts,
    how='inner',
    predicate='within'
)

dropped = len(fire_gdf) - len(joined)
pct     = dropped / len(fire_gdf) * 100
print(f"Before join : {len(fire_gdf):,}")
print(f"After join  : {len(joined):,}")
print(f"Dropped     : {dropped:,} ({pct:.2f}%)")
print("(Dropped points fall on borders, water bodies, or GADM polygon gaps — expected.)")


Before join : 1,242,007
After join  : 1,179,053
Dropped     : 62,954 (5.07%)
(Dropped points fall on borders, water bodies, or GADM polygon gaps — expected.)


In [17]:
# Add crop season label 
season_map = {4: 'Rabi', 5: 'Rabi', 10: 'Kharif', 11: 'Kharif'}
joined['season'] = joined['month'].map(season_map)

# Rename GADM columns and select output columns
# 'frp' (Fire Radiative Power, MW) is kept — required by NB 04.
joined = joined.rename(columns={'NAME_1': 'state', 'NAME_2': 'district'})
out_cols = ['latitude', 'longitude', 'acq_date', 'year', 'season', 'state', 'district', 'frp']
joined = joined[out_cols]

print(f"\nSeason distribution:")
print(joined['season'].value_counts().to_string())
print(f"\nState distribution:")
print(joined['state'].value_counts().to_string())



Season distribution:
season
Kharif    951422
Rabi      227631

State distribution:
state
Punjab     1020759
Haryana     158294


In [18]:
# Save district-mapped fire dataset 
joined.to_csv('../Data/Processed/fire_with_districts.csv', index=False)
print("Saved: data/processed/fire_with_districts.csv")
print(f"   Rows    : {len(joined):,}")
print(f"   Columns : {joined.columns.tolist()}")


✅ Saved: data/processed/fire_with_districts.csv
   Rows    : 1,179,053
   Columns : ['latitude', 'longitude', 'acq_date', 'year', 'season', 'state', 'district', 'frp']
